# Suite VLGPRT — Printing Verilog

The printer writes one fixed layout and the parentheses precedence needs; printing validates the tree and checks
the standard first.

In [ ]:
from mbse.Programs.Framework import Errors
from mbse.Programs.Verilog import SystemVerilog2023, Syntax as S, Verilog2005
from support import raises

P = SystemVerilog2023.print


def name(spelling):
    return S.Identifier(spelling=spelling)


def ref(spelling):
    return S.NameExpression(name=name(spelling))


def number(spelling="1"):
    return S.IntegerLiteral(spelling=spelling)


def binary(left, operator, right):
    return S.BinaryExpression(left=left, operator=operator, right=right)


def roundtrip(text):
    """Text reads and prints back to itself."""
    printed = P(SystemVerilog2023.parse(text))
    assert printed == text, printed
    return printed

## VLGPRT-01 · Parentheses follow IEEE 1800's precedence, and right-associative operators

In [ ]:
a, b, c = ref("a"), ref("b"), ref("c")
assert P(binary(binary(a, "+", b), "*", c)) == "(a + b) * c"
assert P(binary(a, "-", binary(b, "-", c))) == "a - (b - c)" and P(binary(binary(a, "-", b), "-", c)) == "a - b - c"
assert P(binary(a, "->", binary(b, "->", c))) == "a -> b -> c" and P(binary(binary(a, "->", b), "->", c)) == "(a -> b) -> c"
def cond():
    return S.ConditionalExpression(condition=ref("a"), consequence=ref("b"), alternative=ref("c"))


assert P(S.ConditionalExpression(condition=cond(), consequence=ref("b"), alternative=cond())) == "(a ? b : c) ? b : a ? b : c"
assert P(S.UnaryExpression(operator="-", operand=binary(a, "+", b))) == "-(a + b)"
assert P(S.UnaryExpression(operator="~", operand=S.UnaryExpression(operator="-", operand=a))) == "~(-a)"
assert P(S.IndexExpression(value=binary(a, "+", b), index=number())) == "(a + b)[1]"
assert P(S.InsideExpression(value=binary(a, "==", b), set=[number()])) == "(a == b) inside {1}"
assert P(binary(a, "+", S.InsideExpression(value=b, set=[c]))) == "a + (b inside {c})"
assert P(binary(S.AssignmentExpression(target=a, operator="=", value=b), "+", c)) == "(a = b) + c"
assert P(S.IncrementExpression(operator="++", operand=binary(a, "+", b))) == "++(a + b)"
assert P(S.CastExpression(type=binary(a, "+", b), value=c)) == "(a + b)'(c)"
print("ok")

## VLGPRT-02 · Every expression and part prints alone

In [ ]:
a = ref("a")
cases = [
    (S.MemberExpression(value=a, member=name("m")), "a.m"), (S.RangeSelect(value=a, left=number("7"), operator=":", right=number("0")), "a[7:0]"),
    (S.RangeSelect(value=a, left=number("0"), operator="+:", right=number("4")), "a[0 +: 4]"),
    (S.RealLiteral(spelling="1.5"), "1.5"), (S.TimeLiteral(spelling="1ns"), "1ns"), (S.UnbasedUnsizedLiteral(value="z"), "'z"),
    (S.StringLiteral(text="hi"), '"hi"'), (S.Concatenation(items=[a, number()]), "{a, 1}"),
    (S.Replication(count=number("2"), items=[a]), "{2{a}}"),
    (S.AssignmentPattern(items=[S.PatternItem(value=number()), S.PatternItem(key=a, value=number("2")), number("3")]), "'{default: 1, a: 2, 3}"),
    (S.AssignmentPattern(type=S.NamedType(name=name("t")), items=[number()]), "t'{1}"),
    (S.CallExpression(callee=a, arguments=[number(), S.NamedConnection(name=name("b"), value=number("2"))]), "a(1, .b(2))"),
    (S.SystemCall(name="$bits", arguments=[S.IntegerAtomType(keyword="int")]), "$bits(int)"), (S.SystemCall(name="$time"), "$time"),
    (S.CastExpression(type=S.ImplicitType(signing="unsigned"), value=a), "unsigned'(a)"),
    (S.MacroUsage(name=name("M"), arguments="1, 2"), "`M(1, 2)"), (S.MacroUsage(name=name("W")), "`W"), (S.DollarExpression(), "$"),
    (S.NameExpression(name=S.ScopedName(scope=name("p"), name=name("x"))), "p::x"),
    (S.RangeDimension(left=number("3"), right=number("0")), "[3:0]"), (S.UnsizedDimension(), "[]"),
    (S.AssociativeDimension(), "[*]"), (S.QueueDimension(bound=number("4")), "[$:4]"), (S.QueueDimension(), "[$]"),
    (S.IntegerVectorType(keyword="bit", signing="signed", dimensions=[S.SizeDimension(size=number("4"))]), "bit signed [4]"),
    (S.ImplicitType(dimensions=[S.RangeDimension(left=number("1"), right=number("0"))]), "[1:0]"),
    (S.EnumType(base=S.IntegerAtomType(keyword="int"), members=[S.EnumMember(name=name("A"), value=number())]), "enum int {A = 1}"),
    (S.StructType(keyword="union", members=[S.StructMember(type=S.KeywordType(keyword="string"), declarators=[S.VariableDeclarator(name=name("s"))])]), "union { string s; }"),
    (S.AnsiPort(name=name("p")), "p"), (S.InterfacePort(name=name("i")), "interface i"), (S.PortReference(name=name("r")), "r"),
    (S.IncludeDirective(path="types.svh", system=True), "`include <types.svh>"),
    (S.TimedStatement(timing=S.DelayControl(value=S.BinaryExpression(left=a, operator="+", right=number()))), "#(a + 1);"),
    (S.VirtualInterfaceType(interface=name("bus"), parameters=[number("4")], modport=name("mp")), "virtual bus #(4).mp"),
    (S.NewExpression(scope=S.ScopedName(scope=name("p"), name=S.ParameterizedName(name=name("c"), parameters=[number()])),
                     arguments=[a]), "p::c #(1)::new(a)"),
    (S.NewCopyExpression(value=S.CallExpression(callee=a)), "new a()"),
    (S.NewCopyExpression(value=S.BinaryExpression(left=a, operator="+", right=number())), "new (a + 1)"),
    (S.ForwardTypedefDeclaration(name=name("t")), "typedef t;"),
    (S.ConstraintPrototype(static=True, name=name("k")), "static constraint k;"),
    (S.NameExpression(name=S.LocalName(name=name("x"))), "local::x"),
    (S.RandomizeWithExpression(call=S.CallExpression(callee=ref("randomize"))), "randomize() with {}"),
    (S.DistExpression(value=a, items=[S.DistItem(value=number())]), "a dist {1}"),
]
for node, text in cases:
    assert P(node) == text, (P(node), text)
print("ok")

## VLGPRT-03 · Layout: blocks open on their header's line, `end else`, one item per line

In [ ]:
roundtrip("""module m;
    always_ff @(posedge clk) begin
        if (a) begin
            b <= 1;
        end else if (c) begin
            b <= 2;
        end else begin
            b <= 3;
        end
        if (a)
            b <= 4;
        else if (c)
            b <= 5;
        else
            b <= 6;
        unique case (a) inside
            1, [2:3]: b <= 1;
            4: begin
                b <= 2;
            end
            default: ;
        endcase
        for (int i = 0; i < 2; i++)
            a = i;
        do begin
        end while (a);
        do
            a = 1;
        while (a);
        wait (a);
        wait (a) b = 1;
        wait (a) begin
        end
        #1 a = 1;
        @(posedge clk) begin
        end
        assert (a);
        assert (a) b = 1; else b = 2;
        assert (a) begin
        end else begin
        end
        assert (a) else $error("x");
    end
endmodule
""")
roundtrip("""module m;
    if (W) begin : a
    end else if (V) begin : b
    end else
        assign x = 1;

    if (W)
        assign x = 2;
    else
        assign x = 3;

    case (W)
        1:
            assign x = 4;
        default: begin
        end
    endcase

    for (genvar i = 0; i < 2; i++) begin : g
    end : g

    for (j = 0; j < 2; j = j + 1)
        assign x = 5;
endmodule
""")
print("ok")

## VLGPRT-04 · Layout: design units, declarations and directives, with blank lines around what spans lines

In [ ]:
roundtrip("""// lead
`timescale 1ns / 1ps

/* the package */
package p;
    typedef struct packed {
        logic a;
        logic b;
    } s;
    var struct packed {
        logic a;
        logic b;
    } v;
    typedef union { int i; } u;

    function automatic int f(input int a = 1, output b);
        return a;
    endfunction : f

    task t;
        input a;
    endtask
endpackage : p

module m (a, b);
    input wire a;
    output reg b;

`ifdef X
    wire x;
`elsif Y
    wire y;
`else
    wire z;
`endif
endmodule

interface i
    import p::*;
#(
    parameter int W = 1,
    parameter type T = logic
) (
    input logic clk
);
    modport mp (input clk);
endinterface

program q #(
    localparam N = 1
);
endprogram
""")
text = P(SystemVerilog2023.parse("// lead\n`timescale 1ns / 1ps\n/* the package */\npackage p; typedef struct packed { logic a; logic b; } s; endpackage\n"))
assert text == "// lead\n`timescale 1ns / 1ps\n\n/* the package */\npackage p;\n    typedef struct packed {\n        logic a;\n        logic b;\n    } s;\nendpackage\n", text
print("ok")

## VLGPRT-05 · Printing validates the tree and checks the standard

In [ ]:
with raises(Errors.PrintError, match="a BinaryExpression needs a left"):
    P(S.BinaryExpression(operator="+", right=number()))
with raises(Errors.PrintError, match="PackageDeclaration is not Verilog, but this is Verilog-2005"):
    Verilog2005.print(S.PackageDeclaration(name=name("p")))
assert Verilog2005.print(S.ModuleDeclaration(keyword="module", name=name("m"))) == "module m;\nendmodule"
assert P(S.Comment(text=" c")) == "// c" and P(S.Comment(block=True, text=" c ")) == "/* c */"
print("ok")

## VLGPRT-06 · Layout: trailing comments, conditionals among statements, bodies that span lines

In [ ]:
roundtrip("""module m;  // trailing
    wire a;  /* after a */

    initial begin
        x = (a inside {1, 2}) && b;

`ifdef A
        x = 1;
`else
        x = 2;
`endif

        case (a)
            1:
                if (b)
                    x = 1;
                else
                    x = 2;
            default: ;
        endcase
        assert (a) $display("pass");
        assert (a) begin
            x = 1;
        end else $error("fail");
        ;
    end
endmodule
""")
print("ok")


## VLGPRT-07 · Layout: every option of headers, ports, declarations and statements

In [ ]:
roundtrip("""`define EMPTY

module a;
endmodule  // after a

// before b
module b
    import p::*;
(x, y);
    input var logic x;
    output y;
endmodule

module c
    import p::*;
(
    output var logic z = 1'b0,
    w
);
    wire #3 n;
    wire #(d + 1) m;
    const var static int k = 1;
    var v;
    static struct {
        logic a;
        logic b;
    } s;
    struct {
        logic a;
        logic b;
    } u;
    const struct packed {
        logic a;
        logic b;
    } [1:0] t = '0;
    typedef struct packed { logic a; } one;
    typedef enum {A} e;

    /* two
       lines */
    function int f(input var int i, j);
        return 1;
    endfunction

    task r;
        return;
    endtask

    initial begin
        ->> ev;
        disable named;
        disable fork;
        unique if (a)
            b = 1;
        for (; ; );
        for (int k = 0, m = 1; ; );
        #(a + 1) b = 1;
    end
endmodule
""")
print("ok")


## VLGPRT-08 · Layout: classes, their parameters, bases, members and prototypes

In [ ]:
roundtrip("""typedef class driver;

virtual class base #(
    type T = int,
    int N = 2
) extends parent #(T) implements i1, i2;
    local const static int count = 0;
    extern virtual function void f(int x);
    pure virtual task t;

    function new(int x = 0);
        super.new(x);
    endfunction
endclass : base

class derived extends base #(byte, 3)(1, .x(2));
endclass

interface class i1 extends j, k;
    pure virtual function void h;
endclass

module m #(
    int W = 8,
    type U = logic
);
    wire struct packed {
        logic a;
        logic b;
    } w;
endmodule
""")
print("ok")


## VLGPRT-09 · Layout: constraints, inline constraints and `randcase`

In [ ]:
roundtrip("""class c;
    rand int q[];

    constraint k {  // first
        x > 0;
        soft y < 10;
        x -> {
            y == 1;
        }
        x ->
            y == 2;
        if (x)
            y > 2;
        else if (z) {
            y < 3;
        } else
            y == 4;
        foreach (q[i]) {
            q[i] < 5;
        }
        solve x before y, q;
        disable soft y;
        unique {x, [1:2]};
        x dist {1 := 2, [3:4] :/ 5, 7, default :/ 1};

`ifdef SIM
        x < 9;
`endif
    }

    extern static constraint e;

    function void f;
        ok = randomize() with { // inline
x -> y; if (a) { b; } else c; if (d) e; foreach (q[i]) q[i] > 0; solve a before b; disable soft x; unique {a, b}; };
        ok = obj.randomize(x) with (x) {};
        ok = randomize() with {
`ifdef SIM
x > 1;
`endif
y < 3; };
        r = q.find(item) with (item > 1);
        randcase
            1: a = 1;
            2: begin
            end
        endcase
        void'(obj.randomize());
    endfunction
endclass

constraint c::e {
    x != y;
}
""")
print("ok")
